In [1]:
import pandas as pd
import requests
import os

In [ ]:
ENV = "STAGE"       # Change to DEV / STAGE / DEMO / PROD
filepath = ""   # Update your file name

In [6]:
if ENV == 'DEV':
    BASEURL = "https://api-dev.platform.wellytics.health/auth/api"
    ADMINID = "1caa1665-ff18-4247-9560-cf3ef2713dec"
elif ENV in ['STAGE', 'DEMO']:
    BASEURL = "https://api-stage.platform.wellytics.health/auth/api"
    ADMINID = "b97a2b09-65e9-4498-a8c9-d5a1d67bd314"
elif ENV == 'PROD':
    BASEURL = "https://api.platform.wellytics.health/auth/api"
    ADMINID = "beeb3418-4d76-47e2-984d-5a075218cb27"
else:
    raise ValueError("Set correct ENV")

authenticate_api = f"{BASEURL}/authenticate"
payload = {"username": "superadmin@wellytics.health", "password": "superadmin@123"}
headers = {
    "Content-Type": "application/json"
}
response = requests.post(authenticate_api, json=payload, headers=headers)
AUTHTOKEN = (response.json()["result"]["token"])

print("Using ENV:", ENV)
print("Base URL:", BASEURL)
print("Admin ID:", ADMINID)


Using ENV: STAGE
Base URL: https://api-stage.platform.wellytics.health/auth/api
Admin ID: b97a2b09-65e9-4498-a8c9-d5a1d67bd314


In [7]:
df = pd.read_csv(filepath, encoding='utf-8')
df = df.fillna('')
df.columns = [c.strip() for c in df.columns]

df = df.rename(columns={
    'First Name': 'firstName',
    'Last Name': 'lastName',
    'Contact Number': 'contactNumber',
    'Email Id': 'email',
    'Organization Id': 'organizationId',
    'Role': 'role',
    'Registration ID': 'registrationId',
    'Speciality': 'speciality',
    'Department': 'department',
    'Designation': 'designation',
    'Password': 'Password'
})

print("Loaded File:")
print(df)


Loaded File:
  firstName  lastName                     email  contactNumber    Password  \
0     Jacob    Nelson   jacob.nelson@example.in     3017465892   Admin@123   
1      Lily  Peterson  lily.peterson@example.in     3128956743   Admin@123   
2     Ethan    Rogers   ethan.rogers@example.in     3237849516  Doctor@123   
3     Grace    Hughes   grace.hughes@example.in     3346587912  Doctor@123   
4     Isaac    Foster   isaac.foster@example.in     3457198624  Doctor@123   
5     Chloe     Price    chloe.price@example.in     3568947215  Doctor@123   
6      Owen      Ward      owen.ward@example.in     3671584927  Doctor@123   
7    Sophia      Bell    sophia.bell@example.in     3789462153  Doctor@123   
8      Ryan       Cox       ryan.cox@example.in     3892546718  Doctor@123   
9      Ella   Griffin   ella.griffin@example.in     3906718425   Admin@123   

                         organizationId             role  registrationId  \
0  ab7ef592-f6f8-4691-b0d4-ed1ae6e13888  Medical Off

In [8]:
class AdminProfileCreation:
    def __init__(self, base_url, api_key=None):
        self.base_url = base_url.rstrip("/")
        self.session = requests.Session()
        if api_key:
            self.session.headers.update({"Authorization": f"Bearer {api_key}"})

    def profile_register(self, regType, firstName, lastName, email, countryCode, contactNumber, organizationId):
        url = f"{self.base_url}/users/internal/register"
        payload = {
            "regType": regType,
            "firstName": firstName,
            "lastName": lastName,
            "email": email,
            "countryCode": countryCode,
            "contactNumber": contactNumber,
            "organizationId": organizationId
        }
        r = self.session.post(url, json=payload)
        print("[REGISTER]", r.json())
        r.raise_for_status()
        return r.json()

    def sales_verification(self, userName, userId, contactNumber):
        url = f"{self.base_url}/salesVerification"
        payload = {
            "username": userName,
            "accountApproved": True,
            "userId": userId,
            "countryCode": "+91",
            "contactNumber": contactNumber
        }
        r = self.session.post(url, json=payload)
        print("[VERIFY]", r.json())
        r.raise_for_status()
        return r.json()

    def active_profile(self, userName, password, contactNumber):
        url = f"{self.base_url}/users/active/assign"
        payload = {
            "users": [{
                "username": userName,
                "password": password,
                "contactNumber": contactNumber,
                "countryCode": "+91"
            }]
        }
        r = self.session.post(url, json=payload)
        print("[ACTIVATE]", r.json())
        r.raise_for_status()
        return r.json()

    def AdminProfileCreationApproval(self, firstName, lastName, email, contactNumber, password, organizationId):
        userName = email

        print("\n[STEP 1] Registering...")
        reg = self.profile_register("admin", firstName, lastName, email, "+91", contactNumber, organizationId)
        userId = reg["result"]["id"]

        print("\n[STEP 2] Verifying...")
        ver = self.sales_verification(userName, userId, contactNumber)

        print("\n[STEP 3] Activating...")
        act = self.active_profile(userName, password, contactNumber)

        return reg["statusCode"], ver["statusCode"], act["statusCode"]


class MOProfileCreation(AdminProfileCreation):
    def MedOffProfileCreationApproval(self, firstName, lastName, email, contactNumber, password, organizationId):
        print("\n[STEP 1] Registering...")
        reg = self.profile_register("medicalOfficer", firstName, lastName, email, "+91", contactNumber, organizationId)
        userId = reg["result"]["id"]

        print("\n[STEP 2] Verifying...")
        ver = self.sales_verification(email, userId, contactNumber)

        print("\n[STEP 3] Activating...")
        act = self.active_profile(email, password, contactNumber)

        return reg["statusCode"], ver["statusCode"], act["statusCode"]

In [9]:
class DoctorProfileCreation:
    def __init__(self, base_url, api_key=None):
        self.base_url = base_url.rstrip('/')
        self.session = requests.Session()
        if api_key:
            self.session.headers.update({'Authorization': f'Bearer {api_key}'})

    def profile_register(self, regType, firstName, lastName, email, countryCode, contactNumber, organizationId):
        url = f"{self.base_url}/users/internal/register"
        r = self.session.post(url, json={
            "regType": regType,
            "firstName": firstName,
            "lastName": lastName,
            "email": email,
            "countryCode": countryCode,
            "contactNumber": contactNumber,
            "organizationId": organizationId
        })
        print("[REGISTER]", r.json())
        r.raise_for_status()
        return r.json()

    def fetch_speciality_id(self):
        url = f"{self.base_url}/doctors/specialities"
        r = self.session.get(url)
        return {i["name"]: i["id"] for i in r.json()["result"]}

    def fetch_department_id(self):
        url = f"{self.base_url}/doctors/departments"
        r = self.session.get(url)
        return {i["name"]: i["id"] for i in r.json()["result"]}

    def verify_profile(self, designation, registrationId, specialityId, departmentId, doctorId, organizationId):
        url = f"{self.base_url}/doctors/{doctorId}/verify"
        payload = {
            "designation": designation,
            "registrationId": registrationId,
            "specialityId": specialityId,
            "departmentId": departmentId,
            "organizationId": organizationId
        }
        r = self.session.post(url, json=payload)
        print("[VERIFY]", r.json())
        r.raise_for_status()
        return r.json()

    def sales_verification(self, username, userId, contactNumber, countryCode= "+91"):
        url = f"{self.base_url}/salesVerification"
        payload = {
            "username": username,
            "accountApproved": True,
            "userId": ADMINID,
            "countryCode": countryCode,
            "contactNumber": contactNumber
        }
        r = self.session.post(url, json=payload)
        print("[SALES VERIFY]", r.json())
        r.raise_for_status()
        return r.json()

    def active_profile(self, username, password, contactNumber):
        url = f"{self.base_url}/users/active/assign"
        payload = {"users": [{
            "username": username,
            "password": password,
            "contactNumber": contactNumber,
            "countryCode": "+91"
        }]}
        r = self.session.post(url, json=payload)
        print("[ACTIVATE]", r.json())
        r.raise_for_status()
        return r.json()

    def DoctorProfileCreationApproval(self, firstName, lastName, email, contactNumber, password, registrationId, designation, speciality, department, organizationId, countryCode= "+91", regType = "doctor"):
        userName = email

        print("\n[STEP 1] Registering Profile...")
        register = self.profile_register(regType, firstName, lastName, email, countryCode, contactNumber, organizationId)
        print(register["statusCode"])

        userId = register['result']["id"]

        print("\n[STEP 2 & 3] Fetching Speciality & Department IDs...")
        speciality_dict = self.fetch_speciality_id()
        department_dict = self.fetch_department_id()

        specialityId = speciality_dict[speciality]
        departmentId = department_dict[department]

        print("\n[STEP 4] Verifying Profile...")
        verify = self.verify_profile(designation, registrationId, specialityId, departmentId, userId, organizationId)
        print(verify["statusCode"])

        print("\n[STEP 5] Sales Verifying Profile...")
        sales_verify = self.sales_verification(userName, userId, contactNumber, countryCode)
        print(sales_verify["statusCode"])

        print("\n[STEP 6] Activating Profile...")
        active = self.active_profile(userName, password, contactNumber)
        print(active["statusCode"])

        return(register["statusCode"], verify["statusCode"], sales_verify["statusCode"], active["statusCode"])


In [10]:
admin_uploader = AdminProfileCreation(BASEURL, AUTHTOKEN)
mo_uploader = MOProfileCreation(BASEURL, AUTHTOKEN)
doc_uploader = DoctorProfileCreation(BASEURL, AUTHTOKEN)

admin_success, admin_failed = [], []
mo_success, mo_failed = [], []
doc_success, doc_failed = [], []

for _, row in df.iterrows():
    fullname = f"{row['firstName']} {row['lastName']}"
    role = row["role"].strip().lower()

    print(f"\n===== PROCESSING {fullname} ({role.upper()}) =====")
    if role == "admin":
        try:
            s = admin_uploader.AdminProfileCreationApproval(
                row["firstName"], row["lastName"], row["email"],
                row["contactNumber"], row["Password"], row["organizationId"]
            )
            admin_success.append(fullname)
        except Exception as e:
            admin_failed.append(fullname)
            print("❌ ERROR:", fullname, "->", e)

    elif role in ["medical officer", "mo"]:
        try:
            s = mo_uploader.MedOffProfileCreationApproval(
                row["firstName"], row["lastName"], row["email"],
                row["contactNumber"], row["Password"], row["organizationId"]
            )
            mo_success.append(fullname)
        except Exception as e:
            mo_failed.append(fullname)
            print("❌ ERROR:", fullname, "->", e)

    elif role == "doctor":
        try:
            s = doc_uploader.DoctorProfileCreationApproval(row["firstName"], row["lastName"], row["email"], row["contactNumber"], row["Password"], row["registrationId"], row["designation"], row["speciality"], row["department"], row["organizationId"])
            doc_success.append(fullname)
        except Exception as e:
            doc_failed.append(fullname)
            print("❌ ERROR:", fullname, "->", e)
      
print("\n===== OVERALL SUMMARY =====")
print(f"Total: {len(admin_success) + len(admin_failed) + len(mo_success) + len(mo_failed) + len(doc_success) + len(doc_failed)}")
print(f"Success: {len(admin_success) + len(mo_success) + len(doc_success)}")
print(f"Failed: {len(admin_failed) + len(mo_failed) + len(doc_failed)}")
print("\n===== ADMIN SUMMARY =====")
print(f"ADMIN: {len(admin_success)} success, {len(admin_failed)} failed")
print("Admin Success:")
for name in admin_success:
    print("-", name)
print("Admin Failed:")
for name in admin_failed:
    print("-", name)
print("\n===== MO SUMMARY =====")
print(f"MO: {len(mo_success)} success, {len(mo_failed)} failed")
print("MO Success:")
for name in mo_success:
    print("-", name)
print("MO Failed:")
for name in mo_failed:
    print("-", name)
print("\n===== DOCTOR SUMMARY =====")
print(f"DOCTOR: {len(doc_success)} success, {len(doc_failed)} failed")
print("Doctor Success:")
for name in doc_success:
    print("-", name)
print("Doctor Failed:")
for name in doc_failed:
    print("-", name)



===== PROCESSING Jacob Nelson (MEDICAL OFFICER) =====

[STEP 1] Registering...
[REGISTER] {'statusCode': 200, 'message': 'Success!! Account has been created, Sales team will get back for further verification', 'success': True, 'result': {'id': 'd3509d75-a395-421d-af43-4c2cb1c4f1db', 'firstName': 'Jacob', 'lastName': 'Nelson', 'username': 'jacob.nelson@example.in', 'contactNumber': 3017465892, 'countryCode': '+91', 'email': 'jacob.nelson@example.in', 'isSalesVerified': 0, 'salesApprovedBy': None, 'salesRejectedBy': None, 'salesApprovedOn': None, 'salesRejectedOn': None, 'salesRejectionReason': None, 'createdTs': '2025-12-11T10:12:08Z', 'updatedTs': '2025-12-11T10:12:08Z', 'createdBy': 'b97a2b09-65e9-4498-a8c9-d5a1d67bd314', 'updatedBy': 'b97a2b09-65e9-4498-a8c9-d5a1d67bd314', 'designation': None, 'registrationId': None, 'inactive': 1, 'isDeleted': 0, 'deletedBy': None, 'deletedOn': None, 'isEmailVerified': 0, 'isMobileVerified': 0, 'botRegId': '0d76d1c5-5c0e-4e07-8011-7efb2e1f22be', 'o